In [ ]:
"""
LSB Audio Steganography
Beginner-friendly version for Google Colab / Jupyter Notebook
"""

import struct
import warnings
from pathlib import Path

import numpy as np


# ---------------------------------------------------------
# WAV READ
# ---------------------------------------------------------

def read_wav_pcm(path):

    raw = Path(path).read_bytes()

    header = raw[:40]

    (dsize,) = struct.unpack_from("<I", raw, 40)

    pcm = np.frombuffer(raw[44:], dtype=np.uint16).copy()

    return header, dsize, pcm


# ---------------------------------------------------------
# WAV WRITE
# ---------------------------------------------------------

def write_wav_pcm(path, header, dsize, pcm):

    Path(path).write_bytes(
        header +
        struct.pack("<I", dsize) +
        pcm.astype(np.uint16).tobytes()
    )


# ---------------------------------------------------------
# DECIMAL TO BITS
# ---------------------------------------------------------

def decimal_to_bits(values, n_bits):

    d = np.asarray(values, dtype=np.float64).reshape(-1, 1)

    power = (2.0 ** np.arange(n_bits)).reshape(1, -1)

    return np.floor(
        np.remainder(d, 2 * power) / power
    ).astype(np.uint8)


# ---------------------------------------------------------
# BITS TO DECIMAL
# ---------------------------------------------------------

def bits_to_decimal(bits):

    if bits.size == 0:
        return np.array([], dtype=np.float64)

    b = np.asarray(bits, dtype=np.float64)

    weights = 2.0 ** np.arange(b.shape[1])

    return (b @ weights).reshape(-1)


# ---------------------------------------------------------
# PASSWORD RANDOMIZER
# ---------------------------------------------------------

def prng(key, length):

    seed = sum(ord(c) * (i + 1) for i, c in enumerate(key))

    rng = np.random.RandomState(seed)

    return (rng.rand(length) > 0.5).astype(np.uint8)


# ---------------------------------------------------------
# SET LSB
# ---------------------------------------------------------

def set_lsb(samples, bits):

    bits = np.asarray(bits, dtype=np.uint16).ravel()

    samples[:] = (samples & np.uint16(0xFFFE)) | bits


# ---------------------------------------------------------
# GET LSB
# ---------------------------------------------------------

def get_lsb(samples):

    return (samples & 1).astype(np.uint8)


# ---------------------------------------------------------
# EMBED MESSAGE
# ---------------------------------------------------------

def embed_message(wavin, wavout, text, password="password123"):

    header, dsize, cover = read_wav_pcm(wavin)

    # Convert message to binary
    bin_arr = decimal_to_bits(
        np.array([ord(c) for c in text], dtype=np.float64),
        8
    )

    m, n = bin_arr.shape

    len_msg = m * n

    # Store message length
    length_bits = decimal_to_bits(
        np.array([m], dtype=np.float64),
        40
    ).reshape(1, -1)

    # Encrypt bits
    bitx = np.bitwise_xor(
        bin_arr.ravel(order="F"),
        prng(password, len_msg)
    )

    binx = bitx.reshape(m, n, order="F")

    # Capacity check
    if cover.size < len_msg + 48:
        raise ValueError("Message too long!")

    # Password checksum
    control = decimal_to_bits(
        np.array([sum(ord(c) for c in password) % 256],
        dtype=np.float64),
        8
    ).reshape(-1)

    # Embed data into LSB
    set_lsb(cover[0:8], control)

    set_lsb(cover[8:48], length_bits.ravel())

    set_lsb(cover[48:48 + len_msg], binx.ravel(order="F"))

    # Save stego WAV
    write_wav_pcm(wavout, header, dsize, cover)

    print("\nMessage embedded successfully!")
    print("Output file:", wavout)


# ---------------------------------------------------------
# EXTRACT MESSAGE
# ---------------------------------------------------------

def extract_message(wavin, password="password123"):

    _, _, stego = read_wav_pcm(wavin)

    # Verify password
    control = get_lsb(stego[0:8])

    expected = sum(ord(c) for c in password) % 256

    if int(bits_to_decimal(control.reshape(1, -1))[0]) != expected:

        warnings.warn("Wrong password or corrupted message!")

        return ""

    # Read message length
    length_bits = get_lsb(stego[8:48]).reshape(1, -1)

    num_chars = int(bits_to_decimal(length_bits)[0])

    len_bits = num_chars * 8

    # Extract bits
    raw_bits = get_lsb(stego[48:48 + len_bits])

    # Decrypt
    dat = np.bitwise_xor(
        raw_bits,
        prng(password, len_bits)
    )

    bin_arr = dat.reshape(num_chars, 8, order="F")

    chars = bits_to_decimal(bin_arr).astype(np.uint8)

    return "".join(chr(c) for c in chars)


# ---------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------

if __name__ == "__main__":

    # WAV filename input
    cover = input("Enter WAV filename: ")

    # Secret message input
    message = input("Enter secret message: ")

    # Password input
    password = input("Enter password: ")

    # Output filename
    stego = "stego_" + cover

    # Check file exists
    if Path(cover).exists():

        # Embed message
        embed_message(cover, stego, message, password)

        # Extract message
        recovered = extract_message(stego, password)

        print("\nRecovered Message:", recovered)

    else:
        print("\nFile not found!")